# Exploratory Data Analysis – Predictive Maintenance of Industrial Machinery
**AICTE Problem Statement #39**

Dataset: [Kaggle – Machine Predictive Maintenance Classification](https://www.kaggle.com/datasets/shivamb/machine-predictive-maintenance-classification)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

print('Libraries loaded successfully')

## 1. Load Dataset

In [ ]:
import config
from src.preprocessing import load_dataset, engineer_features, derive_failure_type_label, inspect_dataset

df_raw = load_dataset()
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

## 2. Dataset Inspection

In [ ]:
summary = inspect_dataset(df_raw)
print('=== Shape ===')
print(summary['shape'])
print('\n=== Missing Values ===')
print(pd.Series(summary['missing_values']))
print(f"\nDuplicate rows: {summary['duplicates']}")

In [ ]:
print('=== Data Types ===')
print(df_raw.dtypes)
print('\n=== Statistical Summary ===')
df_raw.describe().round(3)

## 3. Class Distribution Analysis

In [ ]:
# Binary failure distribution
print('Machine Failure Distribution:')
print(df_raw[config.TARGET_FAILURE].value_counts())
print(f"\nFailure rate: {df_raw[config.TARGET_FAILURE].mean()*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
df_raw[config.TARGET_FAILURE].value_counts().plot.pie(
    labels=['No Failure', 'Failure'], autopct='%1.2f%%',
    colors=['#2ca02c', '#d62728'], ax=ax1
)
ax1.set_title('Machine Failure Distribution')

# Failure type indicators
fail_cols = [c for c in config.FAILURE_TYPE_COLUMNS if c in df_raw.columns]
fail_counts = df_raw[fail_cols].sum().sort_values(ascending=False)
fail_counts.plot.bar(color='#ff7f0e', ax=ax2)
ax2.set_title('Individual Failure Indicator Counts')
ax2.set_xlabel('Failure Type')
ax2.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
df = engineer_features(df_raw)
df = derive_failure_type_label(df)

print('Engineered features added:')
new_cols = [c for c in df.columns if c not in df_raw.columns]
print(new_cols)

print('\nFailure Type Distribution:')
print(df[config.TARGET_FAILURE_TYPE].value_counts())

## 5. Sensor Feature Distributions

In [ ]:
sensor_cols = config.NUMERICAL_FEATURES
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(sensor_cols):
    for label, color in [(0, '#2ca02c'), (1, '#d62728')]:
        axes[i].hist(df[df[config.TARGET_FAILURE]==label][col], bins=40,
                     alpha=0.6, color=color, label='Failure' if label else 'Normal')
    axes[i].set_title(col)
    axes[i].legend()
    axes[i].set_xlabel(col)

# Hide empty subplot
if len(sensor_cols) < len(axes):
    for j in range(len(sensor_cols), len(axes)):
        axes[j].set_visible(False)

plt.suptitle('Sensor Feature Distributions by Failure Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
corr_cols = sensor_cols + ['Temp_Diff', 'Power_W', config.TARGET_FAILURE]
available = [c for c in corr_cols if c in df.columns]
corr = df[available].corr()

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', ax=ax,
            vmin=-1, vmax=1, linewidths=0.4)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with target
print('\nCorrelation with Machine Failure:')
print(corr[config.TARGET_FAILURE].sort_values(ascending=False))

## 7. Box Plots – Sensor by Failure Status

In [ ]:
fig, axes = plt.subplots(1, len(sensor_cols), figsize=(20, 5))
for i, col in enumerate(sensor_cols):
    df.boxplot(column=col, by=config.TARGET_FAILURE, ax=axes[i],
               boxprops=dict(color='#1f77b4'),
               medianprops=dict(color='red'))
    axes[i].set_title(col.split('[')[0].strip(), fontsize=9)
    axes[i].set_xlabel('Machine Failure')

plt.suptitle('Sensor Values by Machine Failure Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Scatter – Torque vs Rotational Speed

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = {0: '#2ca02c', 1: '#d62728'}
for label, grp in df.groupby(config.TARGET_FAILURE):
    ax.scatter(grp['Rotational speed [rpm]'], grp['Torque [Nm]'],
               c=colors[label], alpha=0.4, s=10,
               label='Failure' if label else 'Normal')
ax.set_xlabel('Rotational Speed (rpm)')
ax.set_ylabel('Torque (Nm)')
ax.set_title('Torque vs Rotational Speed')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Engineered Feature: Power Analysis

In [ ]:
if 'Power_W' in df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for label, color in [(0, '#2ca02c'), (1, '#d62728')]:
        ax1.hist(df[df[config.TARGET_FAILURE]==label]['Power_W'], bins=50,
                 alpha=0.6, color=color, label='Failure' if label else 'Normal')
    ax1.set_title('Power Distribution by Failure Status')
    ax1.legend()

    df.boxplot(column='Power_W', by=config.TARGET_FAILURE, ax=ax2)
    ax2.set_title('Power by Failure Status')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

    print('Power statistics by failure:')
    print(df.groupby(config.TARGET_FAILURE)['Power_W'].describe())

## 10. Product Type Analysis

In [ ]:
if 'Type' in df.columns:
    type_failure = df.groupby('Type')[config.TARGET_FAILURE].agg(['sum', 'count', 'mean'])
    type_failure.columns = ['Failures', 'Total', 'Failure Rate']
    type_failure['Failure Rate %'] = (type_failure['Failure Rate'] * 100).round(2)
    print('Failure Rate by Product Type:')
    print(type_failure)

    fig, ax = plt.subplots(figsize=(8, 5))
    type_failure['Failure Rate %'].plot.bar(color=['#1f77b4', '#ff7f0e', '#2ca02c'], ax=ax)
    ax.set_title('Failure Rate by Product Type')
    ax.set_xlabel('Product Type')
    ax.set_ylabel('Failure Rate (%)')
    plt.tight_layout()
    plt.show()

## 11. Key Insights Summary

| Observation | Details |
|---|---|
| **Class Imbalance** | ~3.4% failure rate – requires SMOTE or class weighting |
| **Tool Wear** | Failed machines show higher mean tool wear |
| **Torque** | Higher torque correlates with increased failure risk |
| **Temperature Diff** | Failures often occur at lower temperature differentials (HDF) |
| **Power** | Power anomalies (too high or too low) precede failures |
| **Product Type** | Low-quality (L) products have higher failure rates |
| **TWF Dominance** | Tool Wear Failure is the most common failure type |
